In [ ]:
Sys.time()

### write peaks to bed file and also export raw peaks mtx

In [ ]:
suppressPackageStartupMessages(library(ArchR))
suppressPackageStartupMessages(library(ggplot2))
suppressPackageStartupMessages(library(RColorBrewer))
suppressPackageStartupMessages(library(cowplot))
suppressPackageStartupMessages(library(gridExtra))
suppressPackageStartupMessages(library(ggsci))
suppressPackageStartupMessages(library(scales))
suppressPackageStartupMessages(library(viridis))
suppressPackageStartupMessages(library(tidyverse))
suppressPackageStartupMessages(library(BSgenome))
suppressPackageStartupMessages(library(BSgenome.Hsapiens.UCSC.hg38))
suppressPackageStartupMessages(library(Matrix))

In [ ]:
addArchRGenome("hg38")
addArchRThreads(threads = 8)

In [ ]:
projdir <- '/u/project/cluo/terencew/igvf/2023_YR2/multiome/'
samples <- read.table(paste0(projdir, 'txt/samples.txt'), sep='\t')$V1

### let's actually start with justin peaks

In [ ]:
indir <- '/u/project/cluo/Shared_Datasets/IGVF/202208_Pilot/new_multiome/YS3v6_Data/'
name <- 'Ernst_stateannotate_YS3v6.FullArchR_NamedCluster_DiffPeaks.table.txt.gz'
path <- paste0(indir, name)
peaks <- read.table(path, sep='\t')
dim(peaks)

In [ ]:
head(peaks)

In [ ]:
mask <- 'chrX' != peaks$chr
auto_peaks <- peaks[mask,]
dim(auto_peaks)

In [ ]:
# Remove "chr", convert to numeric, and sort by chrom + start
peaks_clean <- auto_peaks %>%
  mutate(chrom = as.numeric(gsub("chr", "", chr))) %>%
  arrange(chrom, start)
dim(peaks_clean)

In [ ]:
head(peaks_clean)

In [ ]:
select <- c('chr', 'start', 'end')
tmp_peaks <- peaks_clean[,select]
head(tmp_peaks)

In [ ]:
granges <- makeGRangesFromDataFrame(tmp_peaks)
granges

### de novo called peaks

### let's grab a peaks matrix as well

In [ ]:
proj_path <- paste0(projdir, 'atac/archr/processed/YS3_intersect_rna')
proj <- loadArchRProject(path = proj_path, force = FALSE, showLogo = TRUE)

In [ ]:
proj <- addPeakSet(
                  ArchRProj = proj,
                  peakSet = granges,
                  genomeAnnotation = getGenomeAnnotation(proj),
                  force = FALSE
                )

In [ ]:
proj <- addPeakMatrix(proj)

In [ ]:
proj

In [ ]:
getAvailableMatrices(proj)

In [ ]:
mtx <- getMatrixFromProject(
          ArchRProj = proj,
          useMatrix = "PeakMatrix",
          useSeqnames = NULL,
          verbose = TRUE,
          binarize = FALSE,
          threads = getArchRThreads(),
          logFile = createLogFile("getMatrixFromProject")
        )

In [ ]:
mtx

In [ ]:
final_mtx <- t(assay(mtx))
dim(final_mtx)

In [ ]:
out = paste0(projdir, 'mtx/atac/peaks/justin_chromhmm.mtx')
writeMM(final_mtx, out)

In [ ]:
cells <- as.data.frame(mtx@colData)
features <- as.data.frame(mtx@rowRanges)
dim(cells)
dim(features)
head(cells)
head(features)

In [ ]:
out = paste0(projdir, 'mtx/atac/peaks/justin/justin_chromhmm.cells.tsv')
write.table(cells, out, sep='\t', col.names=TRUE, row.names=TRUE, quote=FALSE)

In [ ]:
out = paste0(projdir, 'mtx/atac/peaks/justin/justin_chromhmm.features.tsv')
write.table(features, out, sep='\t', col.names=TRUE, row.names=TRUE, quote=FALSE)